# Tennis Vision — GPU export on Google Colab

Runs the repository's streaming exporter (`scripts/export_web_data.py`: YOLOv8x + ByteTrack, TrackNet, ResNet-50 court keypoints) on a Colab GPU and writes per-frame JSON for the web viewer.

* **Runtime → Change runtime type → GPU** (T4 is enough) before running.
* Everything that costs time is stored on Google Drive: model weights, raw-detection checkpoints and outputs, so a disconnected session resumes where it stopped.
* Put the source video at `MyDrive/tennis_vision/input_videos/tennis_clip.mp4`. Footage is not redistributed with the code.

Resume caveat: the exporter checkpoints every 25 frames. After a resume, player track IDs restart (they are offset so they cannot collide), so a track crossing the resume point is split in two. Prefer finishing in one session.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), "No CUDA device - switch the runtime to GPU before continuing."
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Drive, repository and dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/tennis_vision')
for sub in ('input_videos', 'models', 'outputs'):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
VIDEO = DRIVE / 'input_videos' / 'tennis_clip.mp4'
assert VIDEO.exists(), f'Upload the clip to {VIDEO}'

REPO_URL = 'https://github.com/<your-account>/Tennis-Vision.git'   # your fork with the tracker fix
REPO = pathlib.Path('/content/Tennis-Vision')
if not REPO.exists():
    !git clone --depth 1 {REPO_URL} {REPO}
%cd {REPO}
!pip -q install -e .

## 3. Model weights (cached on Drive)

In [ ]:
import shutil
WEIGHTS = ['keypoints_model_geoaug.pth', 'pose_landmarker_lite.task', 'tracknet.pt']
for w in WEIGHTS:                                   # restore from the Drive cache
    if (DRIVE / 'models' / w).exists() and not (REPO / 'models' / w).exists():
        shutil.copy(DRIVE / 'models' / w, REPO / 'models' / w)
!python scripts/download_models.py || true
if not (REPO / 'models' / 'tracknet.pt').exists():
    # TrackNet weights belong to their author (yastrebksv/TrackNet) and are fetched from there.
    !gdown 1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl -O models/tracknet.pt
if (DRIVE / 'models' / 'yolov8x.pt').exists():
    shutil.copy(DRIVE / 'models' / 'yolov8x.pt', REPO / 'yolov8x.pt')
from ultralytics import YOLO; YOLO('yolov8x.pt')    # downloads once if missing
for w in WEIGHTS:
    shutil.copy(REPO / 'models' / w, DRIVE / 'models' / w)
shutil.copy(REPO / 'yolov8x.pt', DRIVE / 'models' / 'yolov8x.pt')
!ls -la models yolov8x.pt

## 4. Benchmark 40 frames and confirm CUDA is actually used

In [ ]:
import json, time, subprocess
BENCH = DRIVE / 'outputs' / 'bench40'
t0 = time.time()
subprocess.run(['python', 'scripts/export_web_data.py', '--input', str(VIDEO), '--end-frame', '40',
                '--out-dir', str(BENCH), '--tag', 'bench40', '--segment-reason', 'Colab 40-frame benchmark'], check=True)
wall = time.time() - t0
s = json.load(open(BENCH / 'bench40_summary.json'))
print('device:', s['analysis']['device'], '| detector seconds:', s['analysis']['detector_seconds'])
print(f'wall {wall:.1f} s for 40 frames (includes model loading) -> {wall/40:.2f} s/frame upper bound')
assert s['analysis']['device'] == 'cuda', 'Exporter ran on CPU'
det = sum(s['analysis']['detector_seconds'].values()) / 40
print(f'detector time {det:.3f} s/frame -> full court segment (2335 frames) ~ {det*2335/60:.0f} min + overhead')

## 5. Full court segment (frames 0–2334), resumable

In [ ]:
OUT = DRIVE / 'outputs' / 'full'
subprocess.run(['python', 'scripts/export_web_data.py', '--input', str(VIDEO), '--end-frame', '2335',
                '--out-dir', str(OUT), '--tag', 'tennis_clip',
                '--segment-reason', 'continuous court view; frames 2335+ are a celebration close-up and a graphic'],
               check=True)

## 6. Validate the export

In [ ]:
d = json.load(open(OUT / 'tennis_clip_frames.json'))
fps = d['source']['fps']
assert [r['f'] for r in d['frames']] == list(range(len(d['frames'])))
assert all(abs(r['t'] - r['f'] / fps) < 1e-3 for r in d['frames'])
seg = [r for r in d['frames'] if r['in_segment']]
print('frames in segment:', len(seg))
print('court calibration:', d['court']['calibration']['valid'], d['court']['calibration']['median_line_support'])
print('player selection :', d['players']['selection'])
print('ball raw coverage:', d['ball']['raw_coverage'], '| events:', len(d['events']))

## 7. Package for the website (clip + cleaned ball track + movement analytics)

In [ ]:
WEB = DRIVE / 'outputs' / 'web'
subprocess.run(['python', 'scripts/prepare_web_demo.py', '--frames-json', str(OUT / 'tennis_clip_frames.json'),
                '--video', str(VIDEO), '--out', str(WEB)], check=True)
print('Copy', WEB / 'data', 'and', WEB / 'media', 'into the repository web/ folder, then run: python web/serve.py')